## **Stacking Ensemble: 5 Modelos Groq + Meta-Classificador**

**Tarefa 3 do Trabalho Prático — Aprendizagem Profunda**

Stacking = treinar um meta-classificador que aprende a combinar as previsões
dos modelos base de forma ótima, em vez de usar voting fixo.

**Pipeline:**
1. Correr 5 LLMs no **validation** (dataset-samples, 125 textos)
2. Treinar meta-classificador (Logistic Regression) nas previsões + labels reais
3. Correr 5 LLMs no **teste** (dataset-subm2-labels)
4. Meta-classificador decide a classe final no teste

**Modelos base (via Groq):**
1. GPT-OSS 120B
2. Llama 3.3 70B
3. Kimi K2 Instruct
4. Qwen3 32B
5. Llama 4 Scout 17B

**Dados:**
- **Few-shot examples:** dataset-subm1-labels (100 textos) — usados no prompt
- **Validação:** dataset-samples (125 textos) — treino do meta-classificador
- **Teste:** dataset-subm2-labels — avaliação final

### **1. Setup e Dados**

In [ ]:
# !pip install groq

import pandas as pd
import numpy as np
import time
import json
from groq import Groq
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import Counter

sns.set_style('whitegrid')

LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

# ============================================================
# CARREGAR OS 3 DATASETS
# ============================================================

# 1. Few-shot examples (vão no prompt)
df_support = pd.read_csv('../database/dataset-samples.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()

# 2. Validação — treino do meta-classificador
df_val = pd.read_csv('../database/dataset-subm1-labels.csv', sep=';')
df_val.columns = df_val.columns.str.strip().str.lower()

# 3. Teste — avaliação final
df_test = pd.read_csv('../database/dataset-subm2-labels.csv', sep=';')
df_test.columns = df_test.columns.str.strip().str.lower()

print(f'Few-shot (prompt):  {len(df_support)} textos → {df_support["label"].value_counts().to_dict()}')
print(f'Validação (meta):   {len(df_val)} textos → {df_val["label"].value_counts().to_dict()}')
print(f'Teste (final):      {len(df_test)} textos → {df_test["label"].value_counts().to_dict()}')

### **2. Groq: Keys e Modelos**

In [ ]:
GROQ_KEYS = [
    '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',
    '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',
    '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',
    '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',
]

groq_clients = [Groq(api_key=key) for key in GROQ_KEYS]
groq_idx = 0

MODELS = {
    'GPT-OSS-120B':    'openai/gpt-oss-120b',
    'Llama3.3-70B':    'llama-3.3-70b-versatile',
    'Kimi-K2':         'moonshotai/kimi-k2-instruct-0905',
    'Qwen3-32B':       'qwen/qwen3-32b',
    'Llama4-Scout':    'meta-llama/llama-4-scout-17b-16e-instruct',
}

MODEL_NAMES = list(MODELS.keys())
print(f'Groq clients: {len(groq_clients)} keys')
print(f'Modelos: {len(MODELS)}')
for name, mid in MODELS.items():
    print(f'  {name}: {mid}')

### **3. Prompt, Support Set e Funções Base**

In [ ]:
def build_few_shot_prompt(text, support_examples):
    """Few-shot: mostra N exemplos por classe antes de classificar."""
    examples_block = ''
    for _, row in support_examples.iterrows():
        ex_text = row['text'][:400]
        examples_block += f'Text: {ex_text}\nCategory: {row["label"]}\n\n'

    return f"""You are an expert at detecting AI-generated text. Classify the following text into exactly ONE of these categories:

- Human (written by a human, e.g. from Wikipedia)
- Anthropic (generated by Claude)
- Google (generated by Gemini)
- Meta (generated by Llama)
- OpenAI (generated by GPT)

Here are labeled examples:

{examples_block}
Now classify this text. Output ONLY the category name, nothing else.

Text: {text}
Category:"""


N_PER_CLASS = 15
support_set = pd.DataFrame()
for label in LABELS:
    subset = df_support[df_support['label'] == label]
    sampled = subset.sample(min(N_PER_CLASS, len(subset)), random_state=42)
    support_set = pd.concat([support_set, sampled], ignore_index=True)

print(f'Support set: {len(support_set)} exemplos')
for label in LABELS:
    count = len(support_set[support_set['label'] == label])
    print(f'  {label}: {count}')

In [ ]:
def normalize_prediction(raw):
    """Normaliza a resposta do LLM para um dos 5 labels."""
    if not raw or raw.startswith('Error'):
        return None
    raw_lower = raw.strip().lower()
    for label in LABELS:
        if raw_lower == label.lower():
            return label
    for label in LABELS:
        if label.lower() in raw_lower:
            return label
    return None


def ask_groq(prompt, model_id, max_retries=4):
    """Chama o Groq com rotação automática de keys."""
    global groq_idx
    for attempt in range(max_retries):
        client = groq_clients[groq_idx % len(groq_clients)]
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=15,
                temperature=0.0,
            )
            groq_idx += 1
            return response.choices[0].message.content.strip()
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate_limit' in err_str.lower():
                groq_idx += 1
                time.sleep(2)
                continue
            else:
                return f'Error: {e}'
    return 'Error: rate_limit em todas as keys'


def run_groq_model(df_data, model_name, model_id, sleep_seconds=0.5):
    """Corre um modelo Groq em todos os textos."""
    predictions = []
    raw_outputs = []

    for _, row in tqdm(df_data.iterrows(), total=len(df_data), desc=model_name):
        prompt = build_few_shot_prompt(row['text'], support_set)
        raw = ask_groq(prompt, model_id)
        pred = normalize_prediction(raw)
        raw_outputs.append(raw)
        predictions.append(pred)
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    valid = sum(1 for p in predictions if p is not None)
    print(f'  → {valid}/{len(predictions)} válidas')
    return predictions, raw_outputs


def evaluate(predictions, gold_labels, title='', show_plot=True):
    """Avalia previsões. Retorna accuracy e F1 por classe."""
    valid = [(p, g) for p, g in zip(predictions, gold_labels) if p is not None]
    if not valid:
        print('Sem previsões válidas!')
        return 0.0, {}
    preds_clean, golds_clean = zip(*valid)
    acc = sum(p == g for p, g in valid) / len(valid)

    print(f'\n{"=" * 60}')
    print(f'{title} — Accuracy: {acc:.2%} ({len(valid)}/{len(predictions)} válidas)')
    print(f'{"=" * 60}')
    report = classification_report(golds_clean, preds_clean, labels=LABELS,
                                    zero_division=0, output_dict=True)
    print(classification_report(golds_clean, preds_clean, labels=LABELS, zero_division=0))
    f1_per_class = {label: report[label]['f1-score'] for label in LABELS}

    if show_plot:
        cm = confusion_matrix(golds_clean, preds_clean, labels=LABELS)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=LABELS, yticklabels=LABELS)
        plt.xlabel('Previsão'); plt.ylabel('Real')
        plt.title(f'{title} — {acc:.2%}')
        plt.tight_layout(); plt.show()

    return acc, f1_per_class


print('Funções prontas.')

### **4. Fase 1: Correr Modelos no VALIDAÇÃO (125 textos)**

As previsões no set de validação servem para treinar o meta-classificador.

In [ ]:
gold_val = df_val['label'].tolist()

val_preds = {}   # {model_name: [preds]}
val_raw = {}     # {model_name: [raw]}
val_acc = {}     # {model_name: acc}
val_f1 = {}      # {model_name: {classe: f1}}

print('╔' + '═' * 58 + '╗')
print('║  FASE 1: Correr modelos no set de VALIDAÇÃO (125 textos) ║')
print('╚' + '═' * 58 + '╝')

for model_name, model_id in MODELS.items():
    print(f'\n── {model_name} ({model_id}) ──')
    preds, raw = run_groq_model(df_val, model_name, model_id, sleep_seconds=0.5)
    acc, f1 = evaluate(preds, gold_val, f'[VAL] {model_name}', show_plot=False)
    val_preds[model_name] = preds
    val_raw[model_name] = raw
    val_acc[model_name] = acc
    val_f1[model_name] = f1

print(f'\n{"═" * 60}')
print('RESULTADOS INDIVIDUAIS — VALIDAÇÃO')
print(f'{"═" * 60}')
print(f'{"Modelo":<18} {"Accuracy":>10}')
print(f'{"-"*18} {"-"*10}')
for name in MODEL_NAMES:
    print(f'{name:<18} {val_acc[name]:>10.2%}')

### **5. Fase 2: Correr Modelos no TESTE**

In [ ]:
gold_test = df_test['label'].tolist()

test_preds = {}
test_raw = {}
test_acc = {}
test_f1 = {}

print('╔' + '═' * 58 + '╗')
print('║  FASE 2: Correr modelos no set de TESTE                 ║')
print('╚' + '═' * 58 + '╝')

for model_name, model_id in MODELS.items():
    print(f'\n── {model_name} ({model_id}) ──')
    preds, raw = run_groq_model(df_test, model_name, model_id, sleep_seconds=0.5)
    acc, f1 = evaluate(preds, gold_test, f'[TEST] {model_name}', show_plot=False)
    test_preds[model_name] = preds
    test_raw[model_name] = raw
    test_acc[model_name] = acc
    test_f1[model_name] = f1

print(f'\n{"═" * 60}')
print('RESULTADOS INDIVIDUAIS — TESTE')
print(f'{"═" * 60}')
print(f'{"Modelo":<18} {"Val Acc":>10} {"Test Acc":>10}')
print(f'{"-"*18} {"-"*10} {"-"*10}')
for name in MODEL_NAMES:
    print(f'{name:<18} {val_acc[name]:>10.2%} {test_acc[name]:>10.2%}')

### **6. Construir Features para o Meta-Classificador**

Cada amostra vira um vetor de features = one-hot das previsões dos 5 modelos.

Exemplo: se para o texto i, GPT-OSS prevê "Human" e Llama prevê "Meta",
a feature row contém os one-hots de ambas as previsões concatenados → 5×5 = 25 features.

In [ ]:
label_to_idx = {label: i for i, label in enumerate(LABELS)}


def build_meta_features(predictions_dict, model_names, labels):
    """
    Constrói a matriz de features para o meta-classificador.
    
    Para cada amostra, cria um vetor one-hot por modelo (5 classes)
    concatenados → 5 modelos × 5 classes = 25 features.
    
    Também inclui features de "concordância":
    - Quantos modelos concordam na classe mais votada
    - Entropia dos votos (diversidade)
    """
    n_samples = len(predictions_dict[model_names[0]])
    n_labels = len(labels)
    n_models = len(model_names)
    
    # Features: one-hot por modelo + concordância + entropia
    n_features = n_models * n_labels + 2
    X = np.zeros((n_samples, n_features))
    valid_mask = np.ones(n_samples, dtype=bool)
    
    for i in range(n_samples):
        votes = []
        for m_idx, model_name in enumerate(model_names):
            pred = predictions_dict[model_name][i]
            if pred is not None and pred in label_to_idx:
                # One-hot encoding da previsão deste modelo
                offset = m_idx * n_labels
                X[i, offset + label_to_idx[pred]] = 1.0
                votes.append(pred)
            else:
                valid_mask[i] = False
        
        # Features de concordância
        if votes:
            counter = Counter(votes)
            max_agreement = counter.most_common(1)[0][1]
            X[i, -2] = max_agreement / n_models  # Fração de concordância
            
            # Entropia dos votos
            probs = np.array([counter.get(l, 0) / len(votes) for l in labels])
            probs = probs[probs > 0]
            entropy = -np.sum(probs * np.log2(probs))
            X[i, -1] = entropy
    
    return X, valid_mask


# ============================================================
# FEATURES VALIDAÇÃO (treino do meta-classificador)
# ============================================================
X_val, val_mask = build_meta_features(val_preds, MODEL_NAMES, LABELS)
y_val = np.array([label_to_idx.get(g, -1) for g in gold_val])

# Filtrar amostras onde algum modelo não deu previsão
X_val_clean = X_val[val_mask]
y_val_clean = y_val[val_mask]

# ============================================================
# FEATURES TESTE
# ============================================================
X_test, test_mask = build_meta_features(test_preds, MODEL_NAMES, LABELS)
y_test = np.array([label_to_idx.get(g, -1) for g in gold_test])

X_test_clean = X_test[test_mask]
y_test_clean = y_test[test_mask]

# Nomes das features para interpretabilidade
feature_names = []
for m in MODEL_NAMES:
    for l in LABELS:
        feature_names.append(f'{m}→{l}')
feature_names += ['concordância', 'entropia']

print(f'Features por amostra: {X_val.shape[1]}')
print(f'  {len(MODEL_NAMES)} modelos × {len(LABELS)} classes one-hot = {len(MODEL_NAMES) * len(LABELS)}')
print(f'  + 2 features extra (concordância, entropia)')
print(f'\nValidação: {X_val_clean.shape[0]} amostras válidas de {len(gold_val)}')
print(f'Teste:     {X_test_clean.shape[0]} amostras válidas de {len(gold_test)}')

### **7. Treinar Meta-Classificador (Stacking)**

Testa vários meta-classificadores e escolhe o melhor.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# ============================================================
# TESTAR VÁRIOS META-CLASSIFICADORES (CV no validation set)
# ============================================================

meta_classifiers = {
    'LogisticRegression (L2)': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'LogisticRegression (L1)': LogisticRegression(max_iter=1000, C=1.0, penalty='l1', 
                                                   solver='saga', random_state=42),
    'RandomForest':            RandomForestClassifier(n_estimators=100, max_depth=5, 
                                                      random_state=42),
    'GradientBoosting':        GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                          learning_rate=0.1, random_state=42),
    'SVM (RBF)':               SVC(kernel='rbf', C=1.0, random_state=42),
}

print('Cross-Validation (5-fold) no set de validação:')
print(f'{"Meta-Classificador":<28} {"CV Accuracy":>12} {"Std":>8}')
print(f'{"-"*28} {"-"*12} {"-"*8}')

cv_results = {}
for name, clf in meta_classifiers.items():
    # CV com k=5 no validation set
    scores = cross_val_score(clf, X_val_clean, y_val_clean, cv=5, scoring='accuracy')
    cv_results[name] = scores.mean()
    print(f'{name:<28} {scores.mean():>12.2%} {scores.std():>8.3f}')

best_meta_name = max(cv_results, key=cv_results.get)
print(f'\n→ Melhor meta-classificador: {best_meta_name} ({cv_results[best_meta_name]:.2%})')

In [ ]:
# ============================================================
# TREINAR O MELHOR META-CLASSIFICADOR NO VALIDATION COMPLETO
# ============================================================

best_meta = meta_classifiers[best_meta_name]
best_meta.fit(X_val_clean, y_val_clean)

# Accuracy no próprio validation (resubstitution — otimista, mas informativo)
val_resub_acc = best_meta.score(X_val_clean, y_val_clean)
print(f'Meta-classificador: {best_meta_name}')
print(f'Accuracy resubstitution (val): {val_resub_acc:.2%}')

### **8. Aplicar Stacking no Teste**

In [ ]:
# ============================================================
# PREVER NO TESTE COM O META-CLASSIFICADOR
# ============================================================

idx_to_label = {i: label for label, i in label_to_idx.items()}

# Previsões do stacking (só para amostras válidas)
stacking_pred_idx = best_meta.predict(X_test_clean)
stacking_pred_labels_clean = [idx_to_label[p] for p in stacking_pred_idx]

# Reconstruir a lista completa (incluindo Nones para amostras inválidas)
stacking_preds = []
clean_idx = 0
for i in range(len(gold_test)):
    if test_mask[i]:
        stacking_preds.append(stacking_pred_labels_clean[clean_idx])
        clean_idx += 1
    else:
        stacking_preds.append(None)

stacking_acc, stacking_f1 = evaluate(
    stacking_preds, gold_test, 
    f'STACKING ({best_meta_name}) — TESTE'
)

### **9. Baselines: Weighted Voting e Majority Voting no Teste**

In [ ]:
def weighted_voting(predictions_dict, f1_dict, labels):
    """Weighted voting usando F1 do validation como peso."""
    model_names = list(predictions_dict.keys())
    n_samples = len(predictions_dict[model_names[0]])
    preds = []
    for i in range(n_samples):
        scores = {l: 0.0 for l in labels}
        for m in model_names:
            pred = predictions_dict[m][i]
            if pred and pred in f1_dict[m]:
                scores[pred] += f1_dict[m][pred]
        if max(scores.values()) == 0:
            for m in model_names:
                if predictions_dict[m][i]:
                    preds.append(predictions_dict[m][i])
                    break
            else:
                preds.append(None)
        else:
            preds.append(max(scores, key=scores.get))
    return preds


def majority_voting(predictions_dict, labels):
    """Majority voting simples."""
    model_names = list(predictions_dict.keys())
    n_samples = len(predictions_dict[model_names[0]])
    preds = []
    for i in range(n_samples):
        votes = [predictions_dict[m][i] for m in model_names 
                 if predictions_dict[m][i] is not None]
        if votes:
            preds.append(Counter(votes).most_common(1)[0][0])
        else:
            preds.append(None)
    return preds


# Weighted voting no teste (usando F1 do validation como pesos)
wv_preds = weighted_voting(test_preds, val_f1, LABELS)
wv_acc, wv_f1 = evaluate(wv_preds, gold_test, 'Weighted Voting — TESTE')

# Majority voting no teste
mv_preds = majority_voting(test_preds, LABELS)
mv_acc, mv_f1 = evaluate(mv_preds, gold_test, 'Majority Voting — TESTE')

### **10. Comparação Final: Solo vs Voting vs Stacking**

In [ ]:
print('\n' + '═' * 65)
print('COMPARAÇÃO FINAL — TESTE')
print('═' * 65)
print(f'{"Método":<30} {"Val Acc":>10} {"Test Acc":>10}')
print(f'{"-"*30} {"-"*10} {"-"*10}')

for name in MODEL_NAMES:
    print(f'{name:<30} {val_acc[name]:>10.2%} {test_acc[name]:>10.2%}')

print(f'{"-"*30} {"-"*10} {"-"*10}')
print(f'{"Majority Voting":<30} {"—":>10} {mv_acc:>10.2%}')
print(f'{"Weighted Voting (F1)":<30} {"—":>10} {wv_acc:>10.2%}')
print(f'{"STACKING (" + best_meta_name[:15] + ")":<30} {cv_results[best_meta_name]:>10.2%} {stacking_acc:>10.2%}')
print()

best_solo_name = max(test_acc, key=test_acc.get)
all_methods = {
    **test_acc,
    'Majority Voting': mv_acc,
    'Weighted Voting': wv_acc,
    'Stacking': stacking_acc,
}
best_overall_name = max(all_methods, key=all_methods.get)
best_overall_acc = all_methods[best_overall_name]

print(f'Melhor solo:    {best_solo_name} ({test_acc[best_solo_name]:.2%})')
print(f'Melhor global:  {best_overall_name} ({best_overall_acc:.2%})')
print(f'Ganho stacking vs melhor solo: {stacking_acc - test_acc[best_solo_name]:+.2%}')

In [ ]:
# ============================================================
# GRÁFICO COMPARATIVO FINAL
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Accuracy global — todos os métodos
method_labels = MODEL_NAMES + ['Majority Vote', 'Weighted Vote', 'Stacking']
method_accs = [test_acc[n] for n in MODEL_NAMES] + [mv_acc, wv_acc, stacking_acc]
colors = (['#93c5fd'] * len(MODEL_NAMES)) + ['#fcd34d', '#fdba74', '#34d399']

bars = axes[0].barh(method_labels, method_accs, color=colors, height=0.6)
for bar, acc in zip(bars, method_accs):
    axes[0].text(acc + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{acc:.1%}', va='center', fontweight='bold', fontsize=10)
axes[0].set_xlim(0, 1.08)
axes[0].set_xlabel('Accuracy (Teste)')
axes[0].set_title('Accuracy: Solo vs Ensemble vs Stacking', fontsize=13)

# 2. F1 por classe — melhor solo vs stacking
x = np.arange(len(LABELS))
width = 0.2

axes[1].bar(x - width*1.5, [test_f1[best_solo_name].get(l, 0) for l in LABELS],
            width, label=f'{best_solo_name}', color='#93c5fd')
axes[1].bar(x - width*0.5, [mv_f1.get(l, 0) for l in LABELS],
            width, label='Majority', color='#fcd34d')
axes[1].bar(x + width*0.5, [wv_f1.get(l, 0) for l in LABELS],
            width, label='Weighted', color='#fdba74')
axes[1].bar(x + width*1.5, [stacking_f1.get(l, 0) for l in LABELS],
            width, label='Stacking', color='#34d399')

axes[1].set_xticks(x)
axes[1].set_xticklabels(LABELS, rotation=30, ha='right')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1 por Classe (Teste)', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.1)

plt.suptitle('Stacking vs Baselines — Teste Final', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### **11. Interpretabilidade do Meta-Classificador**

In [ ]:
# ============================================================
# PESOS DO META-CLASSIFICADOR (se for Logistic Regression)
# ============================================================

if hasattr(best_meta, 'coef_'):
    print('Coeficientes do meta-classificador (Logistic Regression):')
    print('Valores altos = o meta-classificador confia neste modelo para esta classe.\n')
    
    coefs = best_meta.coef_  # shape: (n_classes, n_features)
    
    fig, ax = plt.subplots(figsize=(16, 6))
    im = ax.imshow(coefs[:, :-2], cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    
    ax.set_yticks(range(len(LABELS)))
    ax.set_yticklabels(LABELS)
    ax.set_xticks(range(len(feature_names) - 2))
    ax.set_xticklabels(feature_names[:-2], rotation=45, ha='right', fontsize=8)
    ax.set_title('Coeficientes do Meta-Classificador\n(azul = negativo, vermelho = positivo)', fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()
    
    # Top features por classe
    print('\nTop-3 features mais importantes por classe:')
    for c_idx, label in enumerate(LABELS):
        top_indices = np.argsort(np.abs(coefs[c_idx, :-2]))[::-1][:3]
        features = [(feature_names[j], coefs[c_idx, j]) for j in top_indices]
        feats_str = ', '.join([f'{name} ({val:+.2f})' for name, val in features])
        print(f'  {label}: {feats_str}')

elif hasattr(best_meta, 'feature_importances_'):
    print('Feature importances (Tree-based):\n')
    importances = best_meta.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    top_n = min(15, len(feature_names))
    ax.barh(range(top_n), importances[sorted_idx[:top_n]], color='#34d399')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]])
    ax.invert_yaxis()
    ax.set_xlabel('Importância')
    ax.set_title('Top Features do Meta-Classificador')
    plt.tight_layout()
    plt.show()

### **12. Análise de Erros do Stacking**

In [ ]:
print('═' * 65)
print('ERROS DO STACKING NO TESTE')
print('═' * 65)

for i, (stack_pred, true) in enumerate(zip(stacking_preds, gold_test)):
    if stack_pred != true:
        texto_preview = df_test.iloc[i]['text'][:80]
        model_votes = {m: test_preds[m][i] for m in MODEL_NAMES}
        votes_str = ' | '.join([f'{m}: {v}' for m, v in model_votes.items() if v])
        
        print(f'  [{df_test.iloc[i]["id"]}] Real: {true} → Stacking: {stack_pred}')
        print(f'    Votos base: {votes_str}')
        print(f'    {texto_preview}...')
        print()

### **13. Exportar**

In [ ]:
df_results = df_test[['id', 'label']].copy()

for name in MODEL_NAMES:
    col = name.replace('-', '_').replace('.', '_').lower()
    df_results[col] = test_preds[name]

df_results['majority_voting'] = mv_preds
df_results['weighted_voting'] = wv_preds
df_results['stacking'] = stacking_preds
df_results['correct_stacking'] = df_results['label'] == df_results['stacking']

df_results.to_csv('stacking_predictions.csv', index=False)
print('Previsões guardadas em stacking_predictions.csv')
df_results.head(10)